# Utilizing Rule-Based Semantic Extraction and Text Parsing to Automate Scoring Cards in *Magic: The Gathering*
### IMT 575 - Spring 2026
Samuel Lee, Sitong Liu, Zach Greenman

In [1]:
import sys
sys.path.insert(0, '/Users/sitt1iu/mtg-mechanics-parser/src')

In [2]:
import pandas as pd
import numpy as np

from mtg_parser.utils.paths import RAW_DIR, PROCESSED_DIR
from mtg_parser.data.extract_creatures import build_creatures_dataset

In [3]:
build_creatures_dataset()

[pipeline] Default-cards dataset already exists.
[pipeline] Creatures dataset already exists.


In [4]:
# Define features we are interested in
SCRYFALL_COLS = [
    'oracle_id',
    'name',
    'set',
    'released_at',
    'cmc',
    'power',
    'toughness',
    'oracle_text',
    'keywords',
    'color_identity',
    'rarity',
]

POINTS_COLS = ['name', 'set', 'set_name', 'points']

# Load data
creatures_full = pd.read_json(RAW_DIR / 'creatures_full.ndjson', lines=True)

combined_points = pd.read_csv(RAW_DIR / 'combined_csv_pts.csv')

combined_points = combined_points[POINTS_COLS]

# Remove duplicates and ensure earliest version of the card for each set
creatures_full['arena_priority'] = (
    creatures_full['arena_id'].notna().astype(int)
)

creatures_full['released_at'] = pd.to_datetime(
    creatures_full['released_at'], errors='coerce'
)

creatures_full = creatures_full.sort_values(
    by=['arena_priority', 'released_at']
)

creatures_full = creatures_full.drop_duplicates(
    subset=['name', 'set'], keep='first'
)

creatures_subset = creatures_full[SCRYFALL_COLS]

# Merge
merged = combined_points.merge(
    creatures_subset,
    on=['name', 'set'],
    how='left',
)

merged['oracle_text'] = merged['oracle_text'].fillna('')

# Add the release year for each card
merged['year'] = merged['released_at'].dt.year

# Get numeric power/toughness values
merged['power_numeric'] = pd.to_numeric(merged['power'], errors='coerce')

merged['toughness_numeric'] = pd.to_numeric(
    merged['toughness'], errors='coerce'
)

column_order = [
    'oracle_id',
    'name',
    'set',
    'set_name',
    'released_at',
    'year',
    'cmc',
    'power',
    'toughness',
    'power_numeric',
    'toughness_numeric',
    'keywords',
    'oracle_text',
    'color_identity',
    'rarity',
    'points',
]

merged = merged[column_order]

# Diagnostics
missing = merged['oracle_id'].isna().sum()

print(f'[cleaning] Missing oracle_id rows: {missing}')

dupes = merged.duplicated(subset=['name', 'set']).sum()

print(f'[cleaning] Duplicate values in rows: {dupes}')

[cleaning] Missing oracle_id rows: 0
[cleaning] Duplicate values in rows: 0


In [5]:
merged.to_csv(
    PROCESSED_DIR / 'combined_points_cleaned.csv',
    index=False
)

print('Saved combined_points_cleaned.csv to ./data/processed')

Saved combined_points_cleaned.csv to ./data/processed


In [6]:
from mtg_parser.features.extractors.triggered import triggered_features
from mtg_parser.features.extractors.activated import activated_features
from mtg_parser.features.extractors.damage import has_direct_damage
from mtg_parser.features.extractors.destroy import has_destroy
from mtg_parser.features.extractors.bounce import has_bounce
from mtg_parser.features.extractors.card_advantage import card_advantage
from mtg_parser.features.extractors.mana_production import mana_production
from mtg_parser.features.extractors.tokens import has_tokens
from mtg_parser.features.extractors.global_effects import has_global
from mtg_parser.features.extractors.reanimate import has_reanimate
from mtg_parser.features.extractors.impulse_draw import has_impulse_draw
from mtg_parser.features.extractors.minus_xx import has_minus_X
from mtg_parser.features.extractors.special import special_features
from mtg_parser.features.extractors.static import static_features
from mtg_parser.features.feature_extractor import AbilityFeaturePipeline
from mtg_parser.pipeline.orchestrator import process_dataset
from mtg_parser.parsing.ability import parse_ability

pipeline = AbilityFeaturePipeline(features=[
    static_features,
    special_features,
    triggered_features,
    has_tokens,
    activated_features,
    has_global,
    card_advantage,
    has_reanimate,
    has_impulse_draw,
    has_destroy,
    has_bounce,
    has_minus_X,
    mana_production,
    has_direct_damage,
])

sample = merged.sample(5)

blocks = process_dataset(sample)


for i, (card_blocks, (_, row)) in enumerate(zip(blocks, sample.iterrows())):
    print(f"\n=== {row['name']} ===")
    print(f"Oracle text: {row['oracle_text']}")
    print(f"Ability blocks: {card_blocks}")
    
    abilities = [parse_ability(b) for b in card_blocks]
    vectors = pipeline.transform(abilities)
    
    for ability, vector in zip(abilities, vectors):
        print(f"  [{ability.type.value}] {ability.raw[:50]}")
        print(f"  → vector: {vector}")


=== Auspicious Starrix ===
Oracle text: Mutate {5}{G} (If you cast this spell for its mutate cost, put it over or under target non-Human creature you own. They mutate into the creature on top plus all abilities from under it.)
Whenever this creature mutates, exile cards from the top of your library until you exile X permanent cards, where X is the number of times this creature has mutated. Put those permanent cards onto the battlefield.
Ability blocks: ['Mutate {5}{G} ', 'Whenever this creature mutates, exile cards from the top of your library until you exile X permanent cards, where X is the number of times this creature has mutated. Put those permanent cards onto the battlefield.']
  [static] Mutate {5}{G}
  → vector: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  [triggered] Whenever this creature mutates, exile cards from t
  → vector: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

=== Mouse Trapper ===
Oracle text: Flash
Valiant — Whenever thi